# AMANAH v0.2 — Kaggle GPU Evaluation & Validation

Run this on a **Kaggle Notebook with Accelerator = GPU** and **Internet = ON**. It does not retrain the model. It pulls the already-trained candidate checkpoint from Hugging Face, runs batched threshold calibration and held-out evaluation on Kaggle's GPU, executes the 35:28 release gate, then uploads the passing artifact to a private `v0.2-validated` Hugging Face branch.


In [ ]:
# 1) GPU + GitHub setup
import os, json, shutil, subprocess, sys
from pathlib import Path
import torch
print('CUDA:',torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('Enable Settings → Accelerator → GPU in Kaggle, then restart the session.')
print('GPU:',torch.cuda.get_device_name(0))
repo=Path('/kaggle/working/Amanah')
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--depth','1','--branch','main','https://github.com/Saahar2001/Amanah-.git',str(repo)],check=True)
project_dir=repo/'Amanah AI model'
assert project_dir.exists(), project_dir
os.chdir(project_dir)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.45,<5','sentencepiece>=0.2,<1','scikit-learn>=1.5','huggingface_hub>=0.35,<2','pydantic>=2.8,<3'],check=True)


In [ ]:
# 2) Read HF_TOKEN securely from Kaggle Secrets and download v0.2-candidate
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, snapshot_download
hf_token=UserSecretsClient().get_secret('HF_TOKEN')
if not hf_token: raise RuntimeError('Create a Kaggle Secret named HF_TOKEN with Hugging Face READ/WRITE access.')
api=HfApi(token=hf_token); identity=api.whoami(); namespace=identity.get('name'); assert namespace
MODEL_REPO=f'{namespace}/semantic-integrity-v0-1'
candidate=Path('/kaggle/working/amanah_candidate')
snapshot_download(repo_id=MODEL_REPO,repo_type='model',revision='v0.2-candidate',token=hf_token,local_dir=str(candidate))
for required in ['model.pt','amanah_config.json','reference_store.json','eval/validation.jsonl','eval/test.jsonl']:
    assert (candidate/required).exists(), required
print('Candidate ready:',candidate)


In [ ]:
# 3) Batched calibration on GPU
subprocess.run([sys.executable,'-m','training.calibrate_thresholds','--checkpoint',str(candidate),'--validation',str(candidate/'eval/validation.jsonl')],check=True)
print('Thresholds:',(candidate/'thresholds.json').read_text())


In [ ]:
# 4) Batched held-out evaluation
evaluation_dir=Path('/kaggle/working/evaluation')
subprocess.run([sys.executable,'-m','training.evaluate','--checkpoint',str(candidate),'--test',str(candidate/'eval/test.jsonl'),'--output',str(evaluation_dir)],check=True)
metrics=json.loads((evaluation_dir/'metrics.json').read_text())
print(json.dumps(metrics,indent=2))
assert metrics.get('status')=='ok', metrics
assert 0.0 <= metrics['coverage'] <= 1.0
assert 0.0 <= metrics['abstention_rate'] <= 1.0


In [ ]:
# 5) Mandatory 35:28 + 2:2 regression gates
sys.path.insert(0,str(candidate))
from handler import EndpointHandler
handler=EndpointHandler(str(candidate))
store=json.loads((candidate/'reference_store.json').read_text(encoding='utf-8'))
row3528=next(x for x in store['ayahs'] if x['ayah_id']=='35:28')
bad=handler({'inputs':{'source_type':'quran','source_ar':row3528['source_ar'],'candidate_en':'And among people and moving creatures and grazing livestock are likewise various colors. Only Allah fears the knowledgeable among His servants. Indeed, Allah is Exalted in Might and Forgiving.','ayah_id':'35:28'}})
print('35:28:',json.dumps(bad,ensure_ascii=False,indent=2))
assert bad['decision'] in {'REVIEW','CRITICAL'}, bad
assert any(d.get('label')=='AGENCY_SHIFT' for d in bad.get('drifts',[])), bad
from amanah_engine.semantic_rules import run_rules
assert 'NEGATION_FLIP' not in {f.label.value for f in run_rules('This is the Book about which there is no doubt.','This is the Book, beyond doubt.')}
print('V0.2 REGRESSION GATES: PASS')


In [ ]:
# 6) Upload a VALIDATED branch only after all gates pass
report={'status':'validated','metrics':metrics,'release_gate_35_28':'PASS','source_branch':'v0.2-candidate'}
(candidate/'VALIDATION_REPORT_V02.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
api.create_branch(repo_id=MODEL_REPO,branch='v0.2-validated',repo_type='model',revision='v0.2-candidate',exist_ok=True,token=hf_token)
info=api.upload_folder(repo_id=MODEL_REPO,repo_type='model',folder_path=str(candidate),revision='v0.2-validated',token=hf_token,commit_message='Validate AMANAH v0.2 on Kaggle GPU')
print('VALIDATED ARTIFACT UPLOADED')
print(f'https://huggingface.co/{MODEL_REPO}/tree/v0.2-validated')
print('Production endpoint has NOT been changed yet.')
